# Agent-as-a-Judge

Same pipeline as `run.py`, cell by cell. Runs against a mock provider when no API key is set.

In [ ]:
import json, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from _common import get_provider, score_batch, agreement, calibrate, apply_calibration
from rubrics import QA_SHORT_ANSWER

provider = get_provider()
print('provider:', provider.name)

In [ ]:
data = json.loads(Path('sample_tasks.json').read_text())
tasks = []
for t in data['tasks']:
    for variant, answer in t['candidates'].items():
        tasks.append({'id': f"{t['id']}_{variant}", 'task': t['task'], 'gold': t['gold'], 'candidate': answer})
print(len(tasks), 'tasks')

In [ ]:
results = score_batch(provider=provider, tasks=tasks, rubric=QA_SHORT_ANSWER)
for jr in results[:6]:
    parts = ' '.join(f'{c}={s:.1f}' for c, s in jr.scores.items())
    print(f'{jr.task_id:<20} {parts}   -> {jr.aggregate():.2f}')

In [ ]:
agr = agreement(results, data['human_labels'], tolerance=0.5)
for k, v in agr.items():
    print(f'{k:<14} {v:.2f}')

In [ ]:
bias = calibrate(results, data['human_labels'])
for k, v in bias.items():
    sign = '+' if v >= 0 else ''
    print(f'{k:<14} {sign}{v:.2f}')

agr2 = agreement(apply_calibration(results, bias), data['human_labels'], tolerance=0.5)
print()
for k, v in agr2.items():
    print(f'{k:<14} {v:.2f}')